## Setup

In [ ]:
# Setup and Initialization
import pandas as pd
import numpy as np
import os
import joblib
import seaborn as sns
import re


# Sklearn metrics for evaluation
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix, classification_report, recall_score, precision_score, log_loss
import statsmodels.api as sm
from scipy import stats
from sklearn.calibration import calibration_curve # Import calibration_curve for decomposition
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
import datetime as dt
import shap
from arch.bootstrap import StationaryBootstrap

# Paths to directories
BASE_PATH = ".."
INTERMEDIATE_PATH = os.path.join(BASE_PATH, "data", "processed")
RESULTS_PATH = os.path.join(BASE_PATH, "results")
OOS_PRED_PATH = os.path.join(RESULTS_PATH, "predictions")
VISUALS_PATH = os.path.join(RESULTS_PATH, "figures")

os.makedirs(VISUALS_PATH, exist_ok=True)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Set horizon

In [ ]:
# Results for horizon
PREDICTION_HORIZON = 3

## Plotting parameters

In [ ]:
# Use a serif font that matches LaTeX's Computer Modern
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.serif'] = 'DejaVu Serif' # Or 'Computer Modern Roman', if installed
mpl.rcParams['text.usetex'] = False # Set to True if you have a full LaTeX installation and want to render math text

# Set font sizes for consistency
mpl.rcParams['font.size'] = 12
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['xtick.labelsize'] = 11
mpl.rcParams['ytick.labelsize'] = 11
mpl.rcParams['legend.fontsize'] = 11
mpl.rcParams['axes.titlesize'] = 14

# Set line widths and styles
mpl.rcParams['axes.linewidth'] = 0.8
mpl.rcParams['lines.linewidth'] = 1.5
mpl.rcParams['patch.linewidth'] = 1.0

# Set grid properties
mpl.rcParams['axes.grid'] = True
mpl.rcParams['grid.linestyle'] = ':'
mpl.rcParams['grid.linewidth'] = 0.5
mpl.rcParams['grid.color'] = '#A9A9A9' # DarkGray

# Set figure properties
mpl.rcParams['figure.figsize'] = (10, 6)
mpl.rcParams['figure.dpi'] = 150
mpl.rcParams['savefig.dpi'] = 300
mpl.rcParams['savefig.format'] = 'pdf'
mpl.rcParams['savefig.bbox'] = 'tight'

# Remove unnecessary spines
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['axes.spines.top'] = False

print("Matplotlib parameters set for LaTeX aesthetic.")

## Loading results, probabilities, and ensembles

In [ ]:
# Load the Out-of-Sample (OOS) Results
print("Step 2: Loading OOS results from the forecasting loop...")
try:
    file_path = os.path.join(OOS_PRED_PATH, f'oos_results_h{PREDICTION_HORIZON}_final_refactored.pkl')
    oos_results = joblib.load(file_path)
    print(f"Successfully loaded results from: {file_path}")
except FileNotFoundError:
    print(f"ERROR: Results file not found at {file_path}. Please run Notebook 2 first.")
    oos_results = None

# Extract the components
if oos_results:
    oos_probs = oos_results['probabilities']
    oos_errors = oos_results['squared_errors']
    oos_actuals = oos_results['actuals']
    oos_loadings = oos_results.get('loadings')

X_untransformed_full = pd.read_pickle(os.path.join(INTERMEDIATE_PATH, 'X_untransformed_monthly.pkl'))
y_target_full = pd.read_pickle(os.path.join(INTERMEDIATE_PATH, 'y_target.pkl'))

In [ ]:
print("\nDataFrame of predicted probabilities...")

# We will create a single DataFrame to easily view and compare the probabilities
all_probs_list = []
for pred_set, models_dict in oos_probs.items():
    for model_name, prob_list in models_dict.items():
        # Create a name for the column
        col_name = f"{pred_set}_{model_name}"
        # Create a pandas Series with the correct dates as the index
        s = pd.Series(prob_list, index=oos_actuals.index, name=col_name)
        all_probs_list.append(s)

# Concatenate all series into a single DataFrame
prob_df = pd.concat(all_probs_list, axis=1)
# Add the actual outcomes for comparison
prob_df['Actual_Recession'] = oos_actuals

## Helper functions

In [ ]:
# Function for Brier Score Decomposition
def brier_score_decomposition(y_true, y_prob, n_bins=10):
    """
    Decomposes the Brier Score into its Reliability, Resolution, and Uncertainty components
    using the standard weighted approach based on binning.
    """
    # Ensure inputs are numpy arrays for calculations
    y_true_np = np.asarray(y_true)
    y_prob_np = np.asarray(y_prob)

    if len(y_true_np) == 0:
         return {'Reliability': np.nan, 'Resolution': np.nan, 'Uncertainty': np.nan}

    # Calculate overall base rate (uncertainty component)
    p_bar = y_true_np.mean()
    uncertainty = p_bar * (1 - p_bar)

    # Use calibration_curve to get mean predicted prob vs mean actual prob in bins
    # It also returns the number of observations in each bin if return_counts=True (requires newer sklearn)
    # We'll manually bin and count for broader compatibility

    # Bin the predicted probabilities
    bins = np.linspace(0., 1. + 1e-8, n_bins + 1)
    bin_indices = np.digitize(y_prob_np, bins)

    reliability = 0
    resolution = 0
    N = len(y_true_np)

    for i in range(1, n_bins + 1):
        bin_mask = (bin_indices == i)
        if not np.any(bin_mask):
            continue

        y_true_bin = y_true_np[bin_mask]
        y_prob_bin = y_prob_np[bin_mask]
        N_k = len(y_true_bin)

        p_bar_k = y_true_bin.mean() # Mean actual outcome in bin k
        p_k = y_prob_bin.mean() # Mean predicted probability in bin k

        # Reliability component (weighted squared difference between actual mean in bin and predicted mean in bin)
        reliability += (N_k / N) * (p_bar_k - p_k)**2

        # Resolution component (weighted squared difference between actual mean in bin and overall actual mean)
        resolution += (N_k / N) * (p_bar_k - p_bar)**2

    return {'Reliability': reliability, 'Resolution': resolution, 'Uncertainty': uncertainty}

In [ ]:
def get_bootstrap_confidence_interval(y_true, y_prob, metric_func, n_bootstraps=1000, ci_level=0.90, horizon=PREDICTION_HORIZON, block_size=round(max(PREDICTION_HORIZON, 420**(1/3)))):
    """
    Calculates a bootstrap confidence interval for a given performance metric
    using the Stationary Bootstrap, which is appropriate for time-series data.
    This version correctly handles the data structure from the 'arch' library.
    """
    print(f"Running Stationary Bootstrap for {metric_func.__name__}...")

    # Align the data and drop NaNs. Convert to NumPy array for the bootstrap.
    data = pd.concat([y_true, y_prob], axis=1).dropna().values

    block_size = block_size

    print(f"Using block size = {block_size}")

    bs = StationaryBootstrap(block_size, data, seed=42)

    bootstrap_scores = []

    # The bootstrap loop
    for i, resampled_data in enumerate(bs.bootstrap(n_bootstraps)):

        resampled_array = resampled_data[0][0]

        # Access the columns by their integer position in the NumPy array
        y_true_resampled = resampled_array[:, 0]
        y_prob_resampled = resampled_array[:, 1]
        # --- END OF CORRECTION ---

        # Handle cases where a bootstrap sample might only contain one class
        if len(np.unique(y_true_resampled)) < 2:
            continue # Skip this bootstrap replication as metrics are undefined

        # Calculate the metric on the resampled data
        score = metric_func(y_true_resampled, y_prob_resampled)
        bootstrap_scores.append(score)

        if (i + 1) % 100 == 0:
            print(f"  ... completed {i + 1}/{n_bootstraps} replications")

    # Calculate the confidence interval from the distribution of bootstrap scores
    if not bootstrap_scores:
        print("Warning: No valid bootstrap scores were generated.")
        return (np.nan, np.nan)

    lower_quantile = (1 - ci_level) / 2
    upper_quantile = 1 - lower_quantile

    lower_bound = np.quantile(bootstrap_scores, lower_quantile)
    upper_bound = np.quantile(bootstrap_scores, upper_quantile)

    print("Bootstrap complete.")
    return (lower_bound, upper_bound, bootstrap_scores)

## Fundamental Metrics

In [ ]:
print("\nCalculating and displaying performance metrics...")

metrics_list = []
y_true_full = oos_actuals

for pred_set, models_dict in oos_probs.items():
    for model_name, prob_list in models_dict.items():

        y_prob = pd.Series(prob_list, index=y_true_full.index)

        # Create boolean mask with datetime index
        valid_idx = y_prob.notna()

        if valid_idx.sum() == 0:
            print(f"Skipping {pred_set}_{model_name} - No valid predictions.")
            continue

        y_true_eval = y_true_full[valid_idx]
        y_prob_eval = y_prob[valid_idx]


        if len(y_true_eval.unique()) < 2:
            print(f"Skipping {pred_set}_{model_name} - Only one class present after dropping NaNs.")
            continue

        pr_auc = average_precision_score(y_true_eval, y_prob_eval)
        roc_auc = roc_auc_score(y_true_eval, y_prob_eval)
        brier = brier_score_loss(y_true_eval, y_prob_eval)
        log_loss_score = log_loss(y_true_eval, y_prob_eval)

        brier_decomp = brier_score_decomposition(y_true_eval, y_prob_eval)

        metrics_list.append({
            'Predictor_Set': pred_set,
            'Model': model_name,
            'PR_AUC': pr_auc,
            'Brier_Score': brier,
            'Resolution': brier_decomp['Resolution'],
            'Reliability': brier_decomp['Reliability'],
            'Uncertainty': brier_decomp['Uncertainty'],
            'ROC_AUC': roc_auc,
            'Log_Loss': log_loss_score,
            'Num_Forecasts': len(y_prob_eval)
        })

results_df = pd.DataFrame(metrics_list)
if not results_df.empty:
    results_df = results_df.sort_values(by='PR_AUC', ascending=False).round(4)

print("\n Out-of-Sample Performance Summary ")
results_df

In [ ]:
# Visualize the Probabilities Over Time
print("\nVisualizing predicted probabilities...")

if not results_df.empty:
    # title variables are what we want displayed on the graph
    benchmark_model = 'Full_XGBoost'
    benchmark_title = 'Full + XGBoost'

    challenger_model = 'Deter_States_Logit_L2'
    challenger_title = 'At Risk + Logit-L2'

    print(f"\nPlotting Benchmark ({benchmark_title}) vs. Top Performer ({challenger_title})")

    fig, ax = plt.subplots(figsize=(10, 3.5))

    # Plot the models' probabilities
    ax.plot(prob_df.index, prob_df[benchmark_model], label=f'{benchmark_title}', color='firebrick')
    ax.plot(prob_df.index, prob_df[challenger_model], label=f'{challenger_title}', color='#1f77b4')


    # Shade the actual NBER recession periods
    y_target_full_for_shading = pd.read_pickle(os.path.join(INTERMEDIATE_PATH, 'y_target.pkl'))

    in_recession = False
    shade_start = None
    for date, value in y_target_full_for_shading['USRECM'].items():
        if value == 1 and not in_recession:
            shade_start = date
            in_recession = True
        elif value == 0 and in_recession:
            ax.axvspan(shade_start, date, color='#696969', alpha=0.2, label='_nolegend_')
            in_recession = False
    if in_recession: # handle recession at the end of the data
        ax.axvspan(shade_start, y_target_full_for_shading.index[-1], color='#696969', alpha=0.2, label='_nolegend_')


    if any(y_target_full_for_shading['USRECM'] == 1):
        ax.fill_between([],[], color='#696969', alpha=0.2, label='Actual Recession (NBER)')


    # ax.set_title(f'OOS Predicted Recession Probabilities (h={PREDICTION_HORIZON} months)', fontsize=10)
    ax.set_ylabel('Probability')
    ax.set_xlabel('Date')
    # ax.legend(loc='upper left')
    ax.grid(True, axis='y', linestyle=':')
    ax.set_ylim(0, 1)

    # Dates to visualize
    ax.set_xlim(prob_df.index.min(), prob_df.index.max())
    # plt.savefig(os.path.join(VISUALS_PATH, f'Zt_vs_Xt_h{PREDICTION_HORIZON}.pdf'), bbox_inches='tight', dpi=300)
    plt.show()
else:
    print("\nNo valid results to visualize.")

In [ ]:
print("\n Confusion Matrix Analysis for Model ")

THRESHOLD = 0.5

model_name = 'Deter_States_Logit_L2'

# Get the probabilities for this model from the prob_df DataFrame
y_prob_model = prob_df[model_name].dropna()

# Align with actuals
y_true_aligned = prob_df['Actual_Recession'].loc[y_prob_model.index]

# Create binary predictions based on the threshold
y_pred_model = (y_prob_model >= THRESHOLD).astype(int)

# Generate and print the confusion matrix
cm = confusion_matrix(y_true_aligned, y_pred_model)
print(f"Confusion Matrix for: {model_name} (Threshold = {THRESHOLD})")
print(cm)

# For better readability, print it with labels
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives (Correctly predicted no recession): {tn}")
print(f"False Positives (Predicted recession, but none occurred): {fp}")
print(f"False Negatives (Predicted no recession, but one occurred): {fn}")
print(f"True Positives (Correctly predicted recession): {tp}")

print("\nFull Classification Report:")
print(classification_report(y_true_aligned, y_pred_model))

In [ ]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt


# Define the models you want to compare
# These names must match the column names in your 'prob_df' DataFrame
MODEL_1_NAME = 'Deter_States_XGBoost' # Your model
MODEL_1_LABEL = 'At Risk + XGBoost'

MODEL_2_NAME = 'PCA_Factors_8_Logit_L1' # The benchmark
MODEL_2_LABEL = 'PCA on Full + XGBoost'

# --- Prepare the Data ---
# Get the true outcomes and the predicted probabilities for each model
y_true = prob_df['Actual_Recession']
y_prob_model_1 = prob_df[MODEL_1_NAME]
y_prob_model_2 = prob_df[MODEL_2_NAME]

# Align all series and drop any rows with missing values
aligned_df = pd.concat([y_true, y_prob_model_1, y_prob_model_2], axis=1).dropna()
y_true_final = aligned_df['Actual_Recession']
y_prob_model_1_final = aligned_df[MODEL_1_NAME]
y_prob_model_2_final = aligned_df[MODEL_2_NAME]


# --- Calculate the points for each curve ---
precision_1, recall_1, thresholds_1 = precision_recall_curve(y_true_final, y_prob_model_1_final)
precision_2, recall_2, thresholds_2 = precision_recall_curve(y_true_final, y_prob_model_2_final)

# --- Get the PR AUC scores for the legend ---
pr_auc_1 = average_precision_score(y_true_final, y_prob_model_1_final)
pr_auc_2 = average_precision_score(y_true_final, y_prob_model_2_final)


# --- Create the Plot ---
plt.figure(figsize=(10, 8))

# Plot the curve for your model
plt.plot(recall_1, precision_1, color='blue', lw=2,
         label=f'{MODEL_1_LABEL} (PR AUC = {pr_auc_1:.3f})')

# Plot the curve for the benchmark model
plt.plot(recall_2, precision_2, color='grey', lw=2, linestyle='--',
         label=f'{MODEL_2_LABEL} (PR AUC = {pr_auc_2:.3f})')

# Plot a reference line for a no-skill classifier
no_skill = len(y_true_final[y_true_final==1]) / len(y_true_final)
plt.plot([0, 1], [no_skill, no_skill], color='red', linestyle=':', label='No-Skill Classifier')

# Formatting
plt.xlabel('Recall (Sensitivity)')
plt.ylabel('Precision')
plt.title(f'Precision-Recall Curve for h={PREDICTION_HORIZON} Months')
plt.legend(loc='lower left')
plt.grid(True, linestyle=':', alpha=0.6)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.show()

## Block Bootstrap

In [ ]:
print("Running Block Bootstrap to Calculate Confidence Intervals")

HORIZON_TO_ANALYZE = PREDICTION_HORIZON
SET_CHAL = 'Full'
MODEL_CHAL = 'XGBoost'
COMBINED_CHAL = SET_CHAL + '_' + MODEL_CHAL

SET_BENCH = 'Deter_States'
MODEL_BENCH = 'Logit_L2'
COMBINED_BENCH = SET_BENCH + '_' + MODEL_BENCH

aligned_df_challenger = prob_df[['Actual_Recession', COMBINED_CHAL, COMBINED_BENCH]].dropna()
y_true_challenger = aligned_df_challenger['Actual_Recession']
y_prob_challenger = aligned_df_challenger[COMBINED_CHAL]
y_true_benchmark = aligned_df_challenger['Actual_Recession']
y_prob_benchmark = aligned_df_challenger[COMBINED_BENCH]

pr_auc_ci_challenger = get_bootstrap_confidence_interval(
    y_true_challenger,
    y_prob_challenger,
    metric_func=average_precision_score,
    n_bootstraps=1000,
    ci_level=0.90,
)

brier_ci_challenger= get_bootstrap_confidence_interval(
    y_true_challenger,
    y_prob_challenger,
    metric_func=brier_score_loss,
    n_bootstraps=1000,
    ci_level=0.90
)

pr_auc_ci_benchmark = get_bootstrap_confidence_interval(
    y_true_benchmark,
    y_prob_benchmark,
    metric_func=average_precision_score,
    n_bootstraps=1000,
    ci_level=0.90,
)

brier_ci_benchmark = get_bootstrap_confidence_interval(
    y_true_benchmark,
    y_prob_benchmark,
    metric_func=brier_score_loss,
    n_bootstraps=1000,
    ci_level=0.90
)

point_estimate_challenger = results_df[(results_df['Model'] == MODEL_CHAL) & (results_df['Predictor_Set'] == SET_CHAL)].iloc[0]
point_estimate_benchmark = results_df[(results_df['Model'] == MODEL_BENCH) & (results_df['Predictor_Set'] == SET_BENCH)].iloc[0]


print("\n" + "="*60)
print(f"      Final Bootstrap Results for h={HORIZON_TO_ANALYZE} (90% Confidence Intervals)")
print("="*60)

print(f"\nChallenger Model: {COMBINED_CHAL}")
print(f"  - PR AUC:   {point_estimate_challenger['PR_AUC']:.3f} (CI: [{pr_auc_ci_challenger[0]:.3f}, {pr_auc_ci_challenger[1]:.3f}])")
print(f"  - Brier:    {point_estimate_challenger['Brier_Score']:.3f} (CI: [{brier_ci_challenger[0]:.3f}, {brier_ci_challenger[1]:.3f}])")

print(f"\nBenchmark Model: {COMBINED_BENCH}")
print(f"  - PR AUC:   {point_estimate_benchmark['PR_AUC']:.3f} (CI: [{pr_auc_ci_benchmark[0]:.3f}, {pr_auc_ci_benchmark[1]:.3f}])")
print(f"  - Brier:    {point_estimate_benchmark['Brier_Score']:.3f} (CI: [{brier_ci_benchmark[0]:.3f}, {brier_ci_benchmark[1]:.3f}])")

challenger_scores_np_pr_auc = np.array(pr_auc_ci_challenger[2])
benchmark_scores_np_pr_auc = np.array(pr_auc_ci_benchmark[2])

challenger_scores_np_brier = np.array(brier_ci_challenger[2])
benchmark_scores_np_brier = np.array(brier_ci_benchmark[2])

challenger_wins_pr_auc = np.sum(challenger_scores_np_pr_auc > benchmark_scores_np_pr_auc)
total_replications_pr_auc = len(challenger_scores_np_pr_auc)
win_percentage_pr_auc = (challenger_wins_pr_auc / total_replications_pr_auc) * 100

challenger_wins_brier = np.sum(challenger_scores_np_brier < benchmark_scores_np_brier)
total_replications_brier = len(challenger_scores_np_brier)
win_percentage_brier = (challenger_wins_brier / total_replications_brier) * 100

print("\n" + "="*60)
print(f"PR AUC: {COMBINED_CHAL} vs. {COMBINED_BENCH}")
print("="*60)
print(f"In {challenger_wins_pr_auc} out of {total_replications_pr_auc} bootstrap replications ({win_percentage_pr_auc:.1f}%),")
print(f"the '{COMBINED_CHAL}' achieved a better PR AUC than the '{COMBINED_BENCH}'.")
print("="*60)

print("\n" + "="*60)
print(f"Brier Score: {COMBINED_CHAL} vs. {COMBINED_BENCH}")
print("="*60)
print(f"In {challenger_wins_brier} out of {total_replications_brier} bootstrap replications ({win_percentage_brier:.1f}%),")
print(f"the '{COMBINED_CHAL}' achieved a better Brier Score than the '{COMBINED_BENCH}'.")
print("="*60)

In [ ]:
HORIZON_TO_ANALYZE = PREDICTION_HORIZON
SET = 'Full'
MODEL = 'XGBoost'
COMBINED = SET + '_' + MODEL
BENCHMARK = 'Deter_States_Logit_L2'

print(f"\nAnalyzing Challenger Model for h={HORIZON_TO_ANALYZE}: {COMBINED}")

aligned_df_challenger = prob_df[['Actual_Recession', COMBINED, BENCHMARK]].dropna()
y_true_challenger = aligned_df_challenger['Actual_Recession']
y_prob_challenger = aligned_df_challenger[COMBINED]
y_true_benchmark = aligned_df_challenger['Actual_Recession']
y_prob_benchmark = aligned_df_challenger[BENCHMARK]

In [ ]:
metric_to_compare = 'PR_AUC'

if metric_to_compare == 'PR_AUC':
  _, _, benchmark_scores = get_bootstrap_confidence_interval(
      y_true_benchmark,
      y_prob_benchmark,
      metric_func=average_precision_score,
      n_bootstraps=1000,
      ci_level=0.90
  )

  _, _, challenger_scores = get_bootstrap_confidence_interval(
      y_true_challenger,
      y_prob_challenger,
      metric_func=average_precision_score,
      n_bootstraps=1000,
      ci_level=0.90
  )
if metric_to_compare == 'Brier_Score':
  _, _, benchmark_scores = get_bootstrap_confidence_interval(
      y_true_benchmark,
      y_prob_benchmark,
      metric_func=brier_score_loss,
      n_bootstraps=1000,
      ci_level=0.90
  )

  _, _, challenger_scores = get_bootstrap_confidence_interval(
      y_true_challenger,
      y_prob_challenger,
      metric_func=brier_score_loss,
      n_bootstraps=1000,
      ci_level=0.90
  )

if metric_to_compare == 'ROC_AUC':
  _, _, benchmark_scores = get_bootstrap_confidence_interval(
      y_true_benchmark,
      y_prob_benchmark,
      metric_func=roc_auc_score,
      n_bootstraps=1000,
      ci_level=0.90
  )

  _, _, challenger_scores = get_bootstrap_confidence_interval(
      y_true_challenger,
      y_prob_challenger,
      metric_func=roc_auc_score,
      n_bootstraps=1000,
      ci_level=0.90
  )

challenger_scores_np = np.array(challenger_scores)
benchmark_scores_np = np.array(benchmark_scores)

if metric_to_compare == 'PR_AUC' or metric_to_compare == 'ROC_AUC':
  challenger_wins = np.sum(challenger_scores_np > benchmark_scores_np)
  total_replications = len(challenger_scores_np)
  win_percentage = (challenger_wins / total_replications) * 100
elif metric_to_compare == 'Brier_Score':
  challenger_wins = np.sum(challenger_scores_np < benchmark_scores_np)
  total_replications = len(challenger_scores_np)
  win_percentage = (challenger_wins / total_replications) * 100

# --- Print the Conclusion ---
print("\n" + "="*60)
print(f"      Head-to-Head Bootstrap Analysis: {COMBINED} vs. {BENCHMARK}")
print("="*60)
print(f"In {challenger_wins} out of {total_replications} bootstrap replications ({win_percentage:.1f}%),")
print(f"the '{COMBINED}' achieved a better {metric_to_compare} than the '{BENCHMARK}'.")
print("="*60)
# print(f"Other way: {100 - win_percentage:.1f}")

In [ ]:
# Use L/2, L, and 2L
base_L = max(PREDICTION_HORIZON, 420**(1/3))
L_grid = [max(1, base_L/2), base_L, base_L*2]

# Store results
ci_lowers = []
ci_uppers = []
midpoints = []

metric_func = brier_score_loss
for L in L_grid:
    print(f"\nRunning bootstrap with block size L={L}")
    lower, upper, _ = get_bootstrap_confidence_interval(
        y_true_challenger,
        y_prob_challenger,
        metric_func,
        n_bootstraps=1000,
        ci_level=0.90,
        block_size=L
    )
    ci_lowers.append(lower)
    ci_uppers.append(upper)
    midpoints.append((lower + upper)/2)

# Plotting
plt.figure(figsize=(8,5))
plt.fill_between(L_grid, ci_lowers, ci_uppers, color='skyblue', alpha=0.5, label='90% CI')
plt.plot(L_grid, midpoints, marker='o', color='blue', label='CI midpoint')
plt.xticks(L_grid)
plt.xlabel("Block length (L)")
plt.ylabel(f"Confidence interval")
plt.title("Sensitivity of bootstrap CI to block length")
plt.legend()
plt.show()


## Forecast Encompassing Test

In [ ]:
print("\n Forecast Encompassing Regressions ")

import statsmodels.api as sm
import pandas as pd
import numpy as np

# Helper Function for Logit Transform
def logit_transform(p):
    """Clamps probabilities and applies the logit transformation."""
    p_clamped = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p_clamped / (1 - p_clamped))

# Function to run a single encompassing test
def run_encompassing_test(y_true, prob_challenger, prob_benchmark, challenger_name, benchmark_name):
    """
    Runs a Probit encompassing regression and prints a formatted conclusion.
    Tests if the challenger forecast encompasses the benchmark forecast.
    """
    print(f"\n Testing if '{challenger_name}' encompasses '{benchmark_name}' ")

    # Transform probabilities to logits
    logit_challenger = logit_transform(prob_challenger)
    logit_benchmark = logit_transform(prob_benchmark)

    # Create the regression design matrix
    X = pd.DataFrame({
        'logit_challenger': logit_challenger,
        'logit_benchmark': logit_benchmark
    })
    X = sm.add_constant(X)

    # Run the Probit regression
    try:
        model = sm.Probit(y_true, X).fit(disp=0)
    except Exception as e:
        print(f"Could not fit Probit model for this pair: {e}")
        return

    challenger_coeff = model.params['logit_challenger']
    benchmark_coeff = model.params['logit_benchmark']

    # Perform and Interpret the Hypothesis Tests
    p_challenger_is_zero = model.pvalues['logit_challenger']
    p_benchmark_is_zero = model.pvalues['logit_benchmark']

    alpha = 0.10

    # Interpretation
    challenger_is_significant = p_challenger_is_zero < alpha
    benchmark_is_significant = p_benchmark_is_zero < alpha

    print(f"   > Challenger ({challenger_name}): Coeff = {challenger_coeff:.3f}, p-value = {p_challenger_is_zero:.4f}")
    print(f"   > Benchmark ({benchmark_name}): Coeff = {benchmark_coeff:.3f}, p-value = {p_benchmark_is_zero:.4f}")

    if challenger_is_significant and not benchmark_is_significant:
        print(f"CONCLUSION: SUCCESS. The '{challenger_name}' forecast ENCOMPASSES the '{benchmark_name}' forecast.")
        print(f"   (p-value for Challenger: {p_challenger_is_zero:.4f}, p-value for Benchmark: {p_benchmark_is_zero:.4f})")
    elif challenger_is_significant and benchmark_is_significant:
        print(f"CONCLUSION: Both forecasts contain unique, valuable information. A combined forecast would be optimal.")
        print(f"   (p-value for Challenger: {p_challenger_is_zero:.4f}, p-value for Benchmark: {p_benchmark_is_zero:.4f})")
    elif not challenger_is_significant and not benchmark_is_significant:
        print(f"CONCLUSION: Test is inconclusive; neither forecast adds significant value to the other.")
        print(f"   (p-value for Challenger: {p_challenger_is_zero:.4f}, p-value for Benchmark: {p_benchmark_is_zero:.4f})")
    else: # not challenger_is_significant and benchmark_is_significant
        print(f"CONCLUSION: FAILURE. The '{benchmark_name}' forecast encompasses the '{challenger_name}' forecast.")
        print(f"   (p-value for Challenger: {p_challenger_is_zero:.4f}, p-value for Benchmark: {p_benchmark_is_zero:.4f})")


challenger = 'Deter_States_Logit_L2'
benchmarks_to_test = ['Full_Logit_L2', 'Full_XGBoost']



# Loop to Run All Comparisons
for benchmark in benchmarks_to_test:
    # Align data for this specific pair and drop any NaNs
    aligned_df = prob_df[['Actual_Recession', challenger, benchmark]].dropna()

    if aligned_df.empty:
        print(f"\nSkipping {benchmark} due to no overlapping data.")
        continue

    y_true_final = aligned_df['Actual_Recession']
    pred_challenger = aligned_df[challenger]
    pred_benchmark = aligned_df[benchmark]

    # Run the test for this challenger-benchmark pair
    run_encompassing_test(y_true_final, pred_challenger, pred_benchmark, challenger, benchmark)

In [ ]:
print("--- Analyzing Forecast Disagreement to Identify Unique Information ---")

# --- Configuration ---
# Use your final, best-performing models
YOUR_MODEL = 'Deter_States_XGBoost' # Or whatever you call your champion
BENCHMARK_MODEL = 'PCA_Factors_8_XGBoost'

# --- Prepare the Data ---
prob_your_model = prob_df[YOUR_MODEL]
prob_benchmark = prob_df[BENCHMARK_MODEL]

# Calculate the difference
disagreement_series = prob_your_model - prob_benchmark

# --- Create the Plot ---
fig, ax = plt.subplots(figsize=(18, 7))

# Plot the disagreement series
ax.plot(disagreement_series.index, disagreement_series, color='purple', label='Prob(Deter) - Prob(PCA)')

# Add a zero line for reference. When the line is here, the models agree.
ax.axhline(0, color='black', linestyle='--')

# Shade the areas where your model was more concerned
ax.fill_between(disagreement_series.index, 0, disagreement_series,
                where=disagreement_series > 0, color='dodgerblue', alpha=0.3,
                label='Deter Model Signals Higher Risk')

# Shade the areas where the PCA model was more concerned
ax.fill_between(disagreement_series.index, 0, disagreement_series,
                where=disagreement_series < 0, color='orangered', alpha=0.3,
                label='PCA Model Signals Higher Risk')

# Add recession shading for context
ax.fill_between(y_target_full.index, ax.get_ylim()[0], ax.get_ylim()[1],
                where=y_target_full['USRECM']==1, color='gray', alpha=0.2,
                transform=ax.get_xaxis_transform(), label='NBER Recession')

# Formatting
ax.set_xlim(prob_df.index.min(), prob_df.index.max())
ax.set_title(f'Forecast Disagreement Plot (h={PREDICTION_HORIZON} Months)', fontsize=16)
ax.set_ylabel('Probability Difference')
ax.set_xlabel('Date')
ax.legend(loc='upper left')
ax.grid(True, linestyle=':', alpha=0.6)
plt.show()

## Feature Importances

In [ ]:
HORIZON_TO_ANALYZE = 3
MODEL_TO_ANALYZE = 'Logit_L2'
PREDICTOR_SET = 'Deter_States'

print(f"--- Analyzing Averaged Feature Importances for h={HORIZON_TO_ANALYZE} ---")
print(f"Model: {MODEL_TO_ANALYZE}, Predictor Set: {PREDICTOR_SET}")

# Construct the correct file path
file_path = os.path.join(RESULTS_PATH, 'predictions', f'oos_results_h{HORIZON_TO_ANALYZE}_final_refactored.pkl')

try:
    oos_results = joblib.load(file_path)
    oos_importances = oos_results.get('importances', {})
    print("Successfully loaded results file.")
except FileNotFoundError:
    print(f"ERROR: Results file not found at {file_path}.")
    oos_importances = None


average_importance = None
if oos_importances:
    try:
        importance_list = oos_importances[PREDICTOR_SET][MODEL_TO_ANALYZE]

        # Filter out any 'None' values that may have occurred if importance calculation failed
        importance_list_cleaned = [s for s in importance_list if isinstance(s, pd.Series)]

        if importance_list_cleaned:
            # Concatenate the list of Series into a single DataFrame.
            # Each row is a time step, each column is a feature.
            importance_df = pd.concat(importance_list_cleaned, axis=1).T

            # Calculate the mean importance for each feature across all time steps
            average_importance = importance_df.mean().sort_values(ascending=False)

            # (Optional but Recommended) Normalize to sum to 1 for percentage contribution
            average_importance = average_importance / average_importance.sum()

            print("Successfully calculated average feature importances.")
        else:
            print("No valid importance Series found for the specified model and predictor set.")

    except KeyError:
        print(f"ERROR: Could not find '{PREDICTOR_SET}' or '{MODEL_TO_ANALYZE}' in the results file.")

# --- 3. Plot the Results ---

if average_importance is not None and not average_importance.empty:
    # Determine the correct x-axis label based on the model type
    if 'Logit' in MODEL_TO_ANALYZE:
        xlabel_text = 'Mean Importance (Absolute Coefficient)'
    elif 'XGBoost' in MODEL_TO_ANALYZE:
        xlabel_text = 'Mean Importance (Gain)'
    else: # RandomForest, HGBoost
        xlabel_text = 'Mean Importance (Gini Impurity)'

    # Plot the top 20 features
    plt.figure(figsize=(10, 8))
    average_importance.head(20).sort_values().plot(kind='barh')
    plt.title(f'Top 20 Average Out-of-Sample Feature Importances (h={HORIZON_TO_ANALYZE})')
    plt.xlabel(xlabel_text)
    plt.show()
else:
    print("Could not generate feature importance plot.")

In [ ]:
if importance_list_cleaned:
    # Row = Time, Col = Feature
    importance_df = pd.concat(importance_list_cleaned, axis=1).T

    # average coef magnitude
    average_importance = importance_df.mean()

    # agg across lags
    lag_summary_df = average_importance.reset_index()
    lag_summary_df.columns = ['FeatureName', 'MeanAbsCoef']
    
    lag_summary_df['BaseFeature'] = lag_summary_df['FeatureName'].apply(
        lambda x: x.split('_lag')[0]
    )
    
    final_table = lag_summary_df.groupby('BaseFeature')['MeanAbsCoef'].sum().sort_values(ascending=False)

    print(f"{'Base Feature':<25} | {'Average Absolute Coefficient':<25}")
    print("-" * 55)
    for feature, val in final_table.head(10).items():
        print(f"{feature:<25} | {val:>25.4f}")

if not final_table.empty:
    plt.figure(figsize=(10, 6))
    
    # Take the top 10 and reverse
    top_10_plot = final_table.head(10).sort_values(ascending=True)
    top_10_plot.plot(kind='barh', color='midnightblue', edgecolor='black', alpha=0.8)
    
    plt.title(f'Top 10 Most Important Predictors (Aggregated Lags)\nFramework: {PREDICTOR_SET} + {MODEL_TO_ANALYZE} (h={HORIZON_TO_ANALYZE})', 
              fontsize=14, fontweight='bold', pad=15)
    
    plt.xlabel('Average Absolute Coefficient', fontsize=12, labelpad=10)
    plt.ylabel('Base Variable', fontsize=12)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    
    # Add values at the end of the bars
    for i, v in enumerate(top_10_plot):
        plt.text(v + (v * 0.01), i, f'{v:.3f}', color='black', va='center', fontweight='semibold')

    plt.tight_layout()
    plt.show()
else:
    print("Error: Could not generate plot because the importance table is empty.")



## Model Confidence Set

In [ ]:
MODELS_FOR_TABLE_1 = {
    'Proposed (Zt, Logit-L2)': ('Deter_States', 'Logit_L2'),
    'Benchmark (Xt, Logit-L2)': ('Full', 'Logit_L2'),
    'Benchmark (PCA of Xt, Logit-L2)': ('PCA_Factors_8', 'Logit_L2'),
    'Benchmark (Xt, XGBoost)': ('Full', 'XGBoost')
}

MODELS_FOR_TABLE_2 = {
    'Disaggregated (Baseline)': ('Deter_States', 'Logit_L2'),
    'Disaggregated + XGBoost': ('Deter_States', 'XGBoost'),
    'Simple Average': ('Deter_Avg', 'Logit_L2'),
    'Simple Average + XGBoost': ('Deter_Avg', 'XGBoost'),
    'PCA on Zt': ('Deter_PCA', 'Logit_L2'),
    'PCA on Zt + XGBoost': ('Deter_PCA', 'XGBoost')
}

horizons = [3, 6, 12]
all_mcs_results = {}

for h in horizons:
    try:
        file_path = os.path.join(OOS_PRED_PATH, f'oos_results_h{h}_final_fixlagged.pkl')
        oos_results = joblib.load(file_path)
        oos_errors = oos_results['squared_errors']
        forecast_dates = oos_results['actuals'].index
    except (FileNotFoundError, KeyError) as e:
        print(f"ERROR: Could not load results for h={h}")
        continue

    all_mcs_results[h] = {}

    loss_series_t1 = {name: pd.Series(oos_errors[pset][model], index=forecast_dates)
                      for name, (pset, model) in MODELS_FOR_TABLE_1.items()}
    losses_df_t1 = pd.DataFrame(loss_series_t1).dropna()
    print(losses_df_t1.mean().sort_values()) # The mean of the losses should be the Brier Score

    mcs_t1 = ModelConfidenceSet(losses_df_t1.values, n_boot=5000, alpha=0.10, show_progress=True)
    mcs_t1.compute()
    results_t1 = mcs_t1.results(as_dataframe=True)
    results_t1.index = losses_df_t1.columns
    all_mcs_results[h]['Table 1'] = results_t1

    loss_series_t2 = {name: pd.Series(oos_errors[pset][model], index=forecast_dates)
                      for name, (pset, model) in MODELS_FOR_TABLE_2.items()}
    losses_df_t2 = pd.DataFrame(loss_series_t2).dropna()

    mcs_t2 = ModelConfidenceSet(losses_df_t2.values, n_boot=5000, alpha=0.10, show_progress=True)
    mcs_t2.compute()
    results_t2 = mcs_t2.results(as_dataframe=True)
    results_t2.index = losses_df_t2.columns
    all_mcs_results[h]['Table 2'] = results_t2

In [ ]:
mcs_dfs = []
for h, tables in all_mcs_results.items():
    for table_name, results_df in tables.items():
        results_df['Horizon'] = h
        results_df['Table'] = table_name
        mcs_dfs.append(results_df)

if mcs_dfs:
    final_mcs_df = pd.concat(mcs_dfs)
    final_mcs_df = final_mcs_df[['Horizon', 'Table', 'pvalues', 'status']].sort_values(by=['Horizon', 'Table', 'pvalues'], ascending=[True, True, False])
    final_mcs_df = final_mcs_df.reset_index().rename(columns={'index': 'Model'})
    display(final_mcs_df)
else:
    print("No MCS results were generated.")
final_mcs_df.to_csv(os.path.join(RESULTS_PATH, 'mcs_t1_t2.csv'), index=False)

## DM Test

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats


print("\n Running DM Test Across a Range of User Preferences (Theta) ")

# Remember, title is what is displayed on graph
challenger_title = 'Deter Ensemble'
benchmark_title = 'PCA Ensemble'

ensemble_method_name = 'Ensemble'

challenger_model_name = 'Deter_States_Logit_L2'
benchmark_model_name = 'Full_XGBoost'

# Prepare the Data
columns_to_align = ['Actual_Recession', challenger_model_name, benchmark_model_name]
df_test = prob_df[columns_to_align].dropna()
y_true_test = df_test['Actual_Recession']
challenger_prob_test = df_test[challenger_model_name]
benchmark_prob_test = df_test[benchmark_model_name]

print(f"Comparing '{challenger_title}' vs. '{benchmark_title}' for h={PREDICTION_HORIZON}")


# Loop Through Thetas and Run the Conditional DM Test
thetas = np.linspace(0.01, 1, 100) # from highly expansion-averse to highly recession-averse (won't plot values lower than 0.5, because the whole point is to predict recessions)

# Initialize empty lists to store the results
p_values = []
dm_stats = []

for theta in thetas:
    # Calculate the asymmetrically weighted loss series for this theta
    loss_challenger = get_asymmetric_loss_series(y_true_test, challenger_prob_test, theta)
    loss_benchmark = get_asymmetric_loss_series(y_true_test, benchmark_prob_test, theta)


    # Run the DM test
    dm_stat, p_val, _ = diebold_mariano_test_robust(
        loss_challenger,
        loss_benchmark,
        h=PREDICTION_HORIZON
    )

    # Append the results to our lists
    p_values.append(p_val) # Store the two-sided p-value
    dm_stats.append(dm_stat) # Store the DM statistic

print("Loop complete. Variables 'thetas', 'p_values', and 'dm_stats' are now populated.")

In [ ]:
dm_results_df = pd.DataFrame({
    'theta': thetas,
    'p_value': p_values,
    'dm_stat': dm_stats
}).set_index('theta')


# Plotting
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']
plt.rcParams['mathtext.fontset'] = 'cm' # Use computer modern for math symbols

fig, ax = plt.subplots(figsize=(16, 9))

# Plot the main p-value line
ax.plot(dm_results_df.index, dm_results_df['p_value'], marker='.', markersize=4, linestyle='-', color='darkgreen', label='p-value of DM test (H₀: Equal Economic Loss)')

# Plot the 10% significance level line
ax.axhline(0.10, color='red', linestyle='--', linewidth=1.5, label='10% Significance Level')

# Region of NO Significant Difference (p > 0.10)
no_sig_mask = dm_results_df['p_value'] > 0.10
ax.fill_between(dm_results_df.index, 0, 1.1, where=no_sig_mask, color='#FDF5E6', alpha=0.6, label='Region of No Significant Difference') # Use a soft, neutral color

# Region where our index is Significantly Superior (p <= 0.10 AND DM stat < 0)
hadi_wins_mask = (dm_results_df['p_value'] <= 0.10) & (dm_results_df['dm_stat'] < 0)
ax.fill_between(dm_results_df.index, 0, 1.1, where=hadi_wins_mask, color='darkseagreen', alpha=0.3, label=f'Region of Significant Difference ({challenger_title} Superior)')

# Region where PCA is Significantly Superior (p <= 0.10 AND DM stat > 0)
benchmark_wins_mask = (dm_results_df['p_value'] <= 0.10) & (dm_results_df['dm_stat'] > 0)
ax.fill_between(dm_results_df.index, 0, 1.1, where=benchmark_wins_mask, color='lightcoral', alpha=0.3, label=f'Region of Significant Difference ({benchmark_title} Superior)')


# Formaating
ax.set_title(f'Statistical Significance of Economic Loss Difference ({challenger_title} vs. {benchmark_title})', fontsize=20, pad=20)
ax.set_xlabel('User Preference ($\\theta$) - Relative Cost of Missing a Recession', fontsize=16)
ax.set_ylabel('Two-Sided P-Value', fontsize=16)
ax.set_ylim(0, 1.05)
ax.set_xlim(0.5, 1.0) # Focus on the relevant range
ax.grid(True, which='major', linestyle=':', linewidth=0.7, color='grey', alpha=0.6)
ax.tick_params(axis='both', which='major', labelsize=12)

# Improve legend clarity
handles, labels = ax.get_legend_handles_labels()
# Define a professional order for the legend
order = [0, 1, 2, 3, 4] if len(labels) == 5 else [0, 1, 2, 3] # Adjust based on how many regions are plotted
ax.legend([handles[idx] for idx in order],[labels[idx] for idx in order], loc='upper right', fontsize=12, frameon=True, fancybox=True, shadow=False, framealpha=0.9)


# Make the plot frame more prominent
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(1)

# plt.savefig(os.path.join(VISUALS_PATH, f'theta_plot_h1_{challenger_title}_{benchmark_title}.pdf'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# COMPANION PLOT: VISUALIZING THE DIEBOLD-MARIANO STATISTIC


if 'dm_stats' in locals() and len(dm_stats) > 0:
    fig, ax = plt.subplots(figsize=(12, 7))

    # Plot the DM statistic over the range of thetas
    ax.plot(thetas, dm_stats, marker='.', linestyle='-', color='indigo',
            label='DM Statistic (LDI Loss - CFNAI Loss)')

    # Add the zero line
    ax.axhline(0, color='black', linestyle='--', linewidth=1.5, label='Line of Equal Performance')


    # Negative values mean our diffusion index is better
    ax.fill_between(thetas, dm_stats, 0, where=[s < 0 for s in dm_stats],
                    color='mediumseagreen', alpha=0.3, interpolate=True,
                    label=f"'{challenger_model_name}' is Superior (Negative Stat)")

    # Positive values mean benchmark is better
    ax.fill_between(thetas, dm_stats, 0, where=[s > 0 for s in dm_stats],
                    color='lightcoral', alpha=0.3, interpolate=True,
                    label=f"'{benchmark_model_name}' is Superior (Positive Stat)")


    # Annotations and Labels
    ax.set_title(f'Relative Performance by User Preference', fontsize=16, pad=20)
    ax.set_xlabel('User Preference (θ) - Relative Cost of Missing a Recession', fontsize=12)
    ax.set_ylabel('Diebold-Mariano Statistic\n(Negative values favor Our Model)', fontsize=12)
    ax.set_xlim(0.5, 0.99)
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend()



    plt.tight_layout()
    plt.show()

else:
    print("Please run the previous DM test loop cell first to generate the 'dm_stats' variable.")

## Diagnostic Sandbox

Possible econ utility test, a lot of considerations though

In [ ]:
!pip install yfinance pandas_datareader -q
import yfinance as yf
import pandas_datareader.data as web
import pandas as pd

#  Define the date range
start_date = '1989-12-01'
end_date = '2024-12-01'

print("Downloading asset price data...")

asset_returns = None # Initialize to None

try:
    #  Download S&P 500 data from Yahoo Finance
    sp500_data = yf.download('^GSPC', start=start_date, end=end_date, progress=False)

    #  Robust Column Selection (This is the corrected part)
    # yfinance column names can change. We check for standard names.
    if 'Adj Close' in sp500_data.columns:
        price_column = 'Adj Close'
    elif 'Close' in sp500_data.columns:
        price_column = 'Close'
    else:
        raise KeyError("Could not find 'Adj Close' or 'Close' in the downloaded S&P 500 data.")

    print(f"Using column '{price_column}' for S&P 500 calculations.")

    # Calculate monthly returns from the selected price column
    sp500_returns = sp500_data[price_column].resample('MS').first().pct_change()
    sp500_returns.name = 'SP'

    #  Download Bond Total Return Index data from FRED
    bond_data = web.DataReader('BAMLCC0A0CMTRIV', 'fred', start_date, end_date)
    bond_returns = bond_data['BAMLCC0A0CMTRIV'].resample('MS').first().pct_change()
    bond_returns.name = 'Bonds'

    #  Combine into a single DataFrame
    asset_returns = pd.concat([sp500_returns, bond_returns], axis=1).dropna()

    print("\\nAsset return data successfully processed.")
    print("Data available from:", asset_returns.index.min().date())
    print(asset_returns.head())

except Exception as e:
    print(f"\\nCould not download or process data. Error: {e}")

### Error diagnostics

In [ ]:
# Prepare Data
model1_name = 'Deter_States_Logit_L2'
model2_name = 'Full_XGBoost'

print(f" Diagnostic: Plotting the difference in forecast probabilities ({model1_name} - {model2_name}) ")

df_diag = prob_df[[model1_name, model2_name]].dropna()
df_diag['Difference'] = df_diag[model1_name] - df_diag[model2_name]


fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(df_diag.index, df_diag['Difference'], color='#1f77b4', label='Prob1 - Prob2')

# Add a zero line for reference
ax.axhline(0, color='black', linestyle='--', linewidth=1)

# Add recession shading for context
y_target_full_for_shading = pd.read_pickle(os.path.join(INTERMEDIATE_PATH, 'y_target.pkl'))
ax.fill_between(y_target_full_for_shading.index, -1, 1,
                where=y_target_full_for_shading['USRECM']==1, color='#696969', alpha=0.1,
                transform=ax.get_xaxis_transform(), label='NBER Recession')


ax.set_ylabel('Probability Difference')
ax.set_xlabel('Date')
ax.set_xlim(prob_df.index.min(), prob_df.index.max())
ax.set_ylim(-0.75, 0.75)
ax.yaxis.set_major_locator(mticker.MultipleLocator(0.25)) # Add this line
# ax.legend(loc='upper left')
ax.grid(True, linestyle=':', alpha=0.6)

# # Highlight key regions
# ax.text(0.02, 0.95, 'Positive Spike = Proposed signals higher recession risk',
#         transform=ax.transAxes, fontsize=10, verticalalignment='top',
#         bbox=dict(boxstyle='round,pad=0.4', fc='lavender', alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join(VISUALS_PATH, f'forecast_disagreement_{PREDICTION_HORIZON}_{model1_name}_{model2_name}.pdf'), bbox_inches='tight', dpi=300)
plt.show()

df_error = prob_df[['Actual_Recession', model1_name, model2_name]].dropna()

# Calculate squared errors for each model at each point in time
df_error['Loss_LDI'] = (df_error['Actual_Recession'] - df_error[model1_name])**2
df_error['Loss_PCA'] = (df_error['Actual_Recession'] - df_error[model2_name])**2

# Split errors
errors_in_recession = df_error[df_error['Actual_Recession'] == 1]
errors_in_expansion = df_error[df_error['Actual_Recession'] == 0]

# Compare SSE in each region
sse_di_recession = errors_in_recession['Loss_LDI'].mean()
sse_benchmark_recession = errors_in_recession['Loss_PCA'].mean()

sse_di_expansion = errors_in_expansion['Loss_LDI'].mean()
sse_benchmark_expansion = errors_in_expansion['Loss_PCA'].mean()

print(f"\n Error Analysis ")
print(f"{'':<20} | {model1_name:<20} | {model2_name:<20}")
print("-" * 65)
print(f"{'MSE during RECESSIONS':<20} | {sse_di_recession:<20.4f} | {sse_benchmark_recession:<20.4f}")
print(f"{'MSE during EXPANSIONS':<20} | {sse_di_expansion:<20.4f} | {sse_benchmark_expansion:<20.4f}")
print("-" * 65)
print(f"{'MSE on Full Sample':<20} | {df_error['Loss_LDI'].mean():<20.4f} | {df_error['Loss_PCA'].mean():<20.4f}")


print("\n Diagnostic Conclusion ")
if sse_di_recession < sse_benchmark_recession:
    print(f"-> Our index makes smaller errors during RECESSION periods.")
else:
    print(f"-> {model2_name} makes smaller errors during RECESSION periods.")

if sse_di_expansion < sse_benchmark_expansion:
    print(f"-> Our index makes smaller errors during EXPANSION periods.")
else:
    print(f"-> {model2_name} makes smaller errors during EXPANSION periods.")

In [ ]:
#  In your Evaluation Notebook
print("\n The Two Dimensions of Forecast Performance ")


models_to_plot = ['Deter_Ensemble', 'PCA_Factors_8_Ensemble', 'ADS_Ensemble', 'Yield_Ensemble']
aligned_df = prob_df[['Actual_Recession'] + models_to_plot].dropna()
y_true_test = aligned_df['Actual_Recession']


theta = 0.9

# Calculate Expansion vs. Recession Loss for Each Model
results = []
for model_name in models_to_plot:
    prob_test = aligned_df[model_name]

    # Calculate asymmetric absolute loss series
    loss_series = get_asymmetric_loss_series(y_true_test, prob_test, theta)

    # Decompose the loss
    expansion_mask = (y_true_test == 0)
    recession_mask = (y_true_test == 1)

    expansion_loss = loss_series[expansion_mask].sum()
    recession_loss = loss_series[recession_mask].sum()

    results.append({
        'Model': model_name,
        'Expansion_Loss': expansion_loss,
        'Recession_Loss': recession_loss
    })

results_df = pd.DataFrame(results)

# Create scatter plot
plt.style.use('seaborn-v0_8-whitegrid') # Use a nice style
fig, ax = plt.subplots(figsize=(10, 8))

for i, row in results_df.iterrows():
    ax.scatter(row['Expansion_Loss'], row['Recession_Loss'], s=200, label=row['Model'], alpha=0.7)


for i, row in results_df.iterrows():
    ax.text(row['Expansion_Loss']*1.01, row['Recession_Loss']*1.01, row['Model'], fontsize=10)


ax.text(0.05, 0.05, 'Ideal Forecast\n(Low Expansion & Recession Loss)',
        transform=ax.transAxes, fontsize=12, ha='left', va='bottom', style='italic',
        bbox=dict(boxstyle='round,pad=0.5', fc='lightgreen', alpha=0.5))


ax.set_title('The Trade-Off Between Expansion Stability and Recession Accuracy', fontsize=16)
ax.set_xlabel('Total Asymmetric Loss During Expansions (Lower is Better)')
ax.set_ylabel('Total Asymmetric Loss During Recessions (Lower is Better)')
ax.legend().set_visible(False)

plt.show()

### Deter States Analysis

In [ ]:
print("Loading analysis-ready datasets...")
y_target_full = pd.read_pickle(os.path.join(INTERMEDIATE_PATH, 'y_target.pkl'))
X_yield_full = pd.read_pickle(os.path.join(INTERMEDIATE_PATH, 'X_yield.pkl'))
X_transformed_full = pd.read_pickle(os.path.join(INTERMEDIATE_PATH, 'X_transformed_monthly.pkl'))
X_untransformed_full = pd.read_pickle(os.path.join(INTERMEDIATE_PATH, 'X_untransformed_monthly.pkl'))
X_ads_full = pd.read_pickle(os.path.join(INTERMEDIATE_PATH, 'X_ads.pkl'))
tcodes = pd.read_pickle(os.path.join(INTERMEDIATE_PATH, 'tcodes.pkl'))

print("All data loaded successfully.")
print(f"Data shape: {X_transformed_full.shape}, Target shape: {y_target_full.shape}")

In [ ]:
VARS_TO_REMOVE = ['ACOGNO', 'TWEXAFEGSMTHx', 'UMCSENTx', 'OILPRICEx']

vars_that_exist_to_remove = [var for var in VARS_TO_REMOVE if var in X_transformed_full.columns]

X_transformed_full = X_transformed_full.drop(columns=vars_that_exist_to_remove)
X_untransformed_full = X_untransformed_full.drop(columns=vars_that_exist_to_remove)

print(f"--- Data Filtering Complete ---")
print(f"Removed {len(vars_that_exist_to_remove)} problematic variables from all master DataFrames.")
print(f"The list of removed variables is: {vars_that_exist_to_remove}")
print(f"New data shape: {X_transformed_full.shape}")

In [ ]:
variable_groups = {
        'Output_Income': ['RPI', 'W875RX1', 'INDPRO', 'IPFPNSS', 'IPFINAL', 'IPCONGD', 'IPDCONGD', 'IPNCONGD', 'IPBUSEQ', 'IPMAT', 'IPDMAT', 'IPNMAT', 'IPMANSICS', 'IPB51222S', 'IPFUELS', 'CUMFNS'],
        'Labor_Market': ['HWI', 'HWIURATIO', 'CLF16OV', 'CE16OV', 'UNRATE', 'UEMPMEAN', 'UEMPLT5', 'UEMP5TO14', 'UEMP15OV', 'UEMP15T26', 'UEMP27OV', 'CLAIMSx', 'PAYEMS', 'USGOOD', 'CES1021000001', 'USCONS', 'MANEMP', 'DMANEMP', 'NDMANEMP', 'SRVPRD', 'USTPU', 'USWTRADE', 'USTRADE', 'USFIRE', 'USGOVT', 'CES0600000007', 'AWOTMAN', 'AWHMAN', 'CES0600000008', 'CES2000000008', 'CES3000000008'],
        'Housing': ['HOUST', 'HOUSTNE', 'HOUSTMW', 'HOUSTS', 'HOUSTW', 'PERMIT', 'PERMITNE', 'PERMITMW', 'PERMITS', 'PERMITW'],
        'Consumption_Orders_Inventories': ['DPCERA3M086SBEA', 'CMRMTSPLx', 'RETAILx', 'AMDMNOx', 'AMDMUOx', 'ANDENOx', 'BUSINVx', 'ISRATIOx'],
        'Money_Credit': ['M1SL', 'M2SL', 'M2REAL', 'BOGMBASE', 'TOTRESNS', 'NONBORRES', 'BUSLOANS', 'REALLN', 'NONREVSL', 'CONSPI', 'DTCOLNVHFNM', 'DTCTHFNM', 'INVEST'],
        'Interest_Rates_Spreads': ['FEDFUNDS', 'CP3Mx', 'TB3MS', 'TB6MS', 'GS1', 'GS5', 'GS10', 'AAA', 'BAA', 'COMPAPFFx', 'TB3SMFFM', 'TB6SMFFM', 'T1YFFM', 'T5YFFM', 'T10YFFM', 'AAAFFM', 'BAAFFM', 'EXSZUSx', 'EXJPUSx', 'EXUSUKx', 'EXCAUSx'],
        'Prices': ['WPSFD49207', 'WPSFD49502', 'WPSID61', 'WPSID62', 'PPICMM', 'CPIAUCSL', 'CPIAPPSL', 'CPITRNSL', 'CPIMEDSL', 'CUSR0000SAC', 'CUSR0000SAD', 'CUSR0000SAS', 'CPIULFSL', 'CUSR0000SA0L2', 'CUSR0000SA0L5', 'PCEPI', 'DDURRG3M086SBEA', 'DNDGRG3M086SBEA', 'DSERRG3M086SBEA'],
        'Stock_Market': ['S&P 500', 'S&P div yield', 'S&P PE ratio', 'VIXCLSx']
    }
def generate_Deter_Indices(X_transformed_train, y_train, horizon, threshold, counter_threshold=None):
    """
    This is the final, unified framework. It uses Ridge (L2) for nowcasting (h<3)
    to retain all signals, and LASSO (L1) for forecasting (h>=3) to perform
    automated feature selection and remove noise.
    Accepts an optional data-driven counter_cyclical_vars list.
    """
    print(f"      -> Generating Deterioration Indices (h={horizon})...")


    counter_cyclical_vars = {'UNRATE', 'UMPMEAN', 'UEMPLT5', 'UEMP5TO14', 'UEMP15OV', 'UEMP15T26', 'UEMP27OV', 'CLAIMSx', 'ISRATIOx', 'VIXCLSx'}

    deterioration_states = pd.DataFrame(index=X_transformed_train.index)
    momentum_signals = pd.DataFrame(index=X_transformed_train.index)


    all_selected_vars = [var for var_list in variable_groups.values() for var in var_list]
    for var in all_selected_vars:
        signal_for_ranking = X_transformed_train[var]
        # Use the counter_cyclical_vars set (either hardcoded or data-driven)
        is_counter_theoretical = var in counter_cyclical_vars
        use_counter_logic = is_counter_theoretical


        # momentum = signal_for_ranking.rolling(window=horizon, min_periods=1).mean()
        input_signal = signal_for_ranking
        # input_signal = signal_for_ranking.ewm(span=3, adjust=False).mean()

        quantile = threshold
        if counter_threshold is not None:
            counter_quantile = counter_threshold
        else:
            counter_quantile = 1 - threshold

        deterioration_threshold = input_signal.quantile(counter_quantile if use_counter_logic else quantile)
        deteriorating_state = pd.Series(0.0, index=input_signal.index)
        if use_counter_logic: deteriorating_state[input_signal > deterioration_threshold] = 1
        else: deteriorating_state[input_signal < deterioration_threshold] = 1
        deterioration_states[var] = deteriorating_state

    return deterioration_states

In [ ]:
print("\n--- Loading Data for Final Analytical Section ---")

HORIZON_TO_ANALYZE = 3

# --- Load the necessary data from your saved results file ---
try:
    file_path = os.path.join(OOS_PRED_PATH, f'oos_results_h{HORIZON_TO_ANALYZE}_final_fixlagged.pkl')
    oos_results = joblib.load(file_path)
    # Use the importances from your champion Zt model
    oos_importances = oos_results['importances']['Deter_States']['Logit_L2']
    forecast_dates = oos_results['actuals'].index
    print("Successfully loaded saved feature importances.")
except (FileNotFoundError, KeyError) as e:
    print(f"ERROR: Could not find required data. Please ensure importances for 'Deter_States' and 'Logit_L2' were saved. Error: {e}")
    oos_importances = None

if oos_importances:
    # --- Create a single, clean DataFrame of the full importance history ---
    # Filter out any None values that may have occurred if a model step failed
    importance_list_cleaned = [s for s in oos_importances if isinstance(s, pd.Series)]
    importance_df = pd.concat(importance_list_cleaned, axis=1).T
    importance_df.index = forecast_dates[:len(importance_df)]

    # =====================================================================================
    #   5.1 The Stable Drivers: A Global Importance Ranking
    # =====================================================================================

    print("\n--- Analyzing 5.1: The Stable, Long-Run Drivers ---")

    # --- Method 1: Aggregate by Base Variable ---
    # Create a mapping from the full feature name (e.g., HOUST_lag6) to its base name (HOUST)
    base_name_map = {col: col.split('_lag')[0] for col in importance_df.columns}

    # Group by the base name and sum the importances of all lags for that variable
    base_variable_importance = importance_df.groupby(base_name_map, axis=1).sum()

    # Calculate the average importance over the entire OOS period
    average_base_importance = base_variable_importance.mean().sort_values(ascending=False)

    # --- Present the Evidence: Top 15 Table ---
    print("\n--- Table X: Top 15 Most Important 'At-Risk' Indicators (1990-2024 Average OOS Importance) ---")
    display(average_base_importance.head(15).to_frame(name='Average Importance').round(4))

    # =====================================================================================
    #   5.2 The Dynamic Drivers: The Evolving Anatomy of Recessions
    # =====================================================================================

    print("\n--- Analyzing 5.2: The Dynamic, Crisis-Specific Drivers ---")

    # --- Define the Pre-Recession Windows ---
    recession_windows = {
        '1990': ('1989-07-01', '1990-06-30'),
        '2001': ('2000-03-01', '2001-02-28'),
        '2008': ('2006-12-01', '2007-11-30'),
        '2020': ('2019-02-01', '2020-01-31')
    }

    # --- Create a mapping from feature (including lags) to economic sector ---
    var_to_group_map = {}
    for group, var_list in variable_groups.items():
        for var in var_list:
            # Also map the lagged versions that exist in the model's feature set
            for lag_suffix in ['', '_lag3', '_lag6', '_lag12']:
                 var_to_group_map[f"{var}{lag_suffix}"] = group

    # --- Aggregate importances by sector ---
    sectoral_importance = importance_df.groupby(var_to_group_map, axis=1).sum()

    # --- Analyze each window ---
    crisis_signatures = {}
    for name, (start, end) in recession_windows.items():
        # Calculate the total importance of all features within the window
        total_importance_in_window = sectoral_importance.loc[start:end].sum().sum()

        # Calculate the average importance of each SECTOR within the window
        avg_sector_importance_in_window = sectoral_importance.loc[start:end].sum()

        # Normalize to get the percentage contribution of each sector
        sector_contribution = (avg_sector_importance_in_window / total_importance_in_window) * 100
        crisis_signatures[name] = sector_contribution

    # --- Present the Final Table ---
    final_table = pd.DataFrame(crisis_signatures)
    final_table.index.name = "Economic Sector"

    print("\n--- Table Y: Sectoral Contribution to Forecasts in Pre-Recession Periods (%) ---")
    display(final_table.round(1))

In [ ]:
print("\n--- Constructing the Horizon-Specific Parsimonious Predictor Sets ---")

# --- This assumes you have run the code to generate and save the importance Series ---
# --- for each horizon in your main analysis notebook.

horizons = [3, 6, 12]
all_parsimonious_sets = {}

print("Loading importance rankings and creating unified sets for each horizon...")

for h in horizons:
    try:
        # --- Load the importance data for this horizon ---
        file_path_h = os.path.join(OOS_PRED_PATH, f'oos_results_h{h}_final_fixlagged.pkl')
        oos_results_h = joblib.load(file_path_h)

        oos_importances_zt_h = oos_results_h['importances']['Deter_States']['Logit_L2']
        oos_importances_xt_h = oos_results_h['importances']['Full']['XGBoost']

        # --- Process Zt importances for this horizon ---
        imp_df_zt = pd.concat([s for s in oos_importances_zt_h if isinstance(s, pd.Series)], axis=1).T
        base_map_zt = {col: col.split('_lag')[0] for col in imp_df_zt.columns}
        avg_imp_zt = imp_df_zt.groupby(base_map_zt, axis=1).sum().mean().sort_values(ascending=False)

        # --- Process Xt importances for this horizon ---
        imp_df_xt = pd.concat([s for s in oos_importances_xt_h if isinstance(s, pd.Series)], axis=1).T
        base_map_xt = {col: col.split('_lag')[0] for col in imp_df_xt.columns}
        avg_imp_xt = imp_df_xt.groupby(base_map_xt, axis=1).sum().mean().sort_values(ascending=False)

        # --- THIS IS THE KEY STEP: Create the horizon-specific unified set ---
        top_zt_vars_h = set(avg_imp_zt.head(10).index)
        top_xt_vars_h = set(avg_imp_xt.head(10).index)

        UNIFIED_SET_H = sorted(list(top_zt_vars_h.union(top_xt_vars_h)))

        all_parsimonious_sets[f'h={h}'] = UNIFIED_SET_H

    except (FileNotFoundError, KeyError) as e:
        print(f"Could not process horizon h={h}. Error: {e}")
        all_parsimonious_sets[f'h={h}'] = ["ERROR: Could not generate set."]

# --- Print the Final, Definitive Sets ---
print("\n" + "="*60)
print("   Definitive Parsimonious Predictor Sets for Final Analysis")
print("="*60)
for horizon, predictor_list in all_parsimonious_sets.items():
    print(f"\n--- Unified Set for {horizon} ({len(predictor_list)} variables) ---")
    print(predictor_list)